In [ ]:
import re
import string
import joblib
import pandas as pd
import numpy as np
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Download required NLTK resources for text processing
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries imported successfully!")

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1081)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1081)>


In [ ]:
#Loading sample security logs (Corpus)
data = {
    "log_text": [
        "CRITICAL: Unauthorized exfiltration attempt detected via outbound SSH tunnel on port 22.",
        "WARNING: Potential privilege escalation observed. User 'admin_guest' executed sudo commands.",
        "INFO: Normal heartbeat connection established successfully from internal gateway.",
        "ERROR: Denial of Service (DoS) attack signature matched. High packet rate flood on port 80.",
        "CRITICAL: Sensitive data dump initiated from database server to external IP."
    ],
    "label": ["Exfiltration", "Privilege Escalation", "Benign", "Denial of Service", "Exfiltration"]
}

df = pd.DataFrame(data)
print(f"Loaded {len(df)} sample log records.")
df.head()

In [ ]:
#Text preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    tokens = text.split()
    cleaned_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens 
        if token not in stop_words
    ]
    
    return " ".join(cleaned_tokens)

# Apply preprocessing to our dataframe
df['cleaned_text'] = df['log_text'].apply(preprocess_text)
df[['log_text', 'cleaned_text']]

In [ ]:
#Feature extraction (TF-IDF)
vectorizer = TfidfVectorizer(max_features=100)
tfidf_matrix = vectorizer.fit_transform(df['cleaned_text'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

In [ ]:
#Train the classification model
X = tfidf_matrix
y = df['label']

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model (using class_weight='balanced' to handle any skews)
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

# Evaluate performance on test data
y_pred = model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:

def predict_threat(new_log, model, vectorizer):
    cleaned = preprocess_text(new_log)
    features = vectorizer.transform([cleaned])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)
    confidence = np.max(probabilities)
    
    return prediction, confidence

# Test log
test_log = "CRITICAL: External IP attempting rapid connection flood on port 443."
pred_label, conf = predict_threat(test_log, model, vectorizer)

print(f"Log: {test_log}")
print(f"Prediction: {pred_label} (Confidence: {conf:.2f})")